# 🇻🇳 Huấn Luyện & Đánh Giá BiLSTM Phân Tích Cảm Xúc Văn Bản Tiếng Việt
### Đề tài 6: Phân tích cảm xúc văn bản tiếng Việt (UIT-VSFC) - Mạng Nơ-ron Hồi quy BiLSTM

> ⚠️ **HƯỚNG DẪN CẤU HÌNH TRƯỚC KHI CHẠY TRÊN KAGGLE:**
> 1. **Bật GPU**: Ở panel *Notebook options* bên phải -> *Accelerator* -> Chọn **GPU T4 x2** hoặc **GPU P100**.
> 2. **Bật Internet**: Trong panel *Notebook options* -> Bật công tắc **Internet ON** (Bắt buộc để tải repo GitHub, dataset UIT-VSFC và PhoW2V).
> 3. *(Tùy chọn)* Cấu hình W&B: Thêm Secret `WANDB_API_KEY` trong mục *Add-ons* -> *Secrets* nếu muốn log trực tiếp lên dashboard.

In [ ]:
# 1. Kiểm tra cấu hình phần cứng GPU
!nvidia-smi

import os, sys, shutil
WORKING_DIR = "/kaggle/working/Vietnamese-Sentiment-Analysis"

# Clone repo hoặc di chuyển vào repo nếu đã tồn tại
if not os.path.exists(WORKING_DIR):
    print("[*] Đang clone kho lưu trữ GitHub...")
    !git clone https://github.com/Perfect-Kopiko/Vietnamese-Sentiment-Analysis.git {WORKING_DIR}
    %cd {WORKING_DIR}
    # Checkout nhánh feat/bilstm-report nếu có, nếu không giữ main
    !git checkout feat/bilstm-report 2>/dev/null || git checkout main 2>/dev/null || echo "Đang ở nhánh mặc định."
else:
    print("[*] Thư mục repo đã tồn tại, chuyển thư mục làm việc...")
    %cd {WORKING_DIR}

# Cài đặt các thư viện bổ sung cần thiết
!pip install -q pyvi underthesea wandb

# Tải bộ dữ liệu UIT-VSFC từ HuggingFace
print("[*] Tải dữ liệu UIT-VSFC...")
!python data/download_data.py

In [ ]:
# 2. Tải Pre-trained Word2Vec tiếng Việt (PhoW2V - 300 chiều)
import os
os.makedirs("data/resources", exist_ok=True)
W2V_FILE = "data/resources/word2vec_vi_words_300dims.txt"

# Kiểm tra xem PhoW2V đã có sẵn từ Kaggle Input Dataset hay chưa
KAGGLE_INPUT_W2V = "/kaggle/input/phow2v/word2vec_vi_words_300dims.txt"
if os.path.exists(KAGGLE_INPUT_W2V):
    print(f"[*] Đã tìm thấy PhoW2V trong Kaggle Input: {KAGGLE_INPUT_W2V}")
    W2V_FILE = KAGGLE_INPUT_W2V
elif not os.path.exists(W2V_FILE):
    print("[*] Đang tải pre-trained PhoW2V từ HuggingFace...")
    W2V_URL = "https://huggingface.co/datasets/uit-nlp/phow2v/resolve/main/word2vec_vi_words_300dims.txt"
    ret = os.system(f"wget -q --show-progress -O {W2V_FILE} {W2V_URL}")
    if ret != 0 or os.path.getsize(W2V_FILE) < 1000:
        print("[!] Chú ý: Nếu đường link HuggingFace bị giới hạn băng thông, bạn có thể tạo Kaggle Dataset chứa PhoW2V và Add Data vào notebook.")
else:
    print(f"[*] File Word2Vec đã tồn tại tại: {W2V_FILE}")

print(f"[*] Đường dẫn Word2Vec sử dụng: {W2V_FILE}")

In [ ]:
# 3. Đăng nhập Weights & Biases (W&B) để theo dõi đồ thị loss & F1
import os
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    wandb_key = user_secrets.get_secret("WANDB_API_KEY")
    import wandb
    wandb.login(key=wandb_key)
    print("[+] Đã đăng nhập W&B thành công với Kaggle Secret!")
except Exception as e:
    print(f"[*] Không tìm thấy Kaggle Secret WANDB_API_KEY ({e}). Tự động kích hoạt chế độ OFFLINE.")
    os.environ["WANDB_MODE"] = "offline"

In [ ]:
# 4. CHẠY CHUỖI THỰC NGHIỆM ABLATION STUDY CHO BILSTM (15 EPOCHS MỖI RUN)
# Tận dụng GPU T4/P100 với batch_size=64

print("\n" + "="*70)
print("RUN 1/6: Đánh giá tác động khi KHÔNG tách từ (Word Segmentation = False)")
print("="*70)
!python scripts/train_bilstm.py \
    --run_name bilstm_no_seg_w2v \
    --use_word_segment false \
    --w2v_path "{W2V_FILE}" \
    --loss_type weighted_ce \
    --epochs 15 \
    --batch_size 64

print("\n" + "="*70)
print("RUN 2/6: Đánh giá tác động khi KHÔNG dùng Word2Vec tiền huấn luyện (Random Embedding)")
print("="*70)
!python scripts/train_bilstm.py \
    --run_name bilstm_seg_no_w2v \
    --use_word_segment true \
    --w2v_path "" \
    --loss_type weighted_ce \
    --epochs 15 \
    --batch_size 64

print("\n" + "="*70)
print("RUN 3/6: So sánh Hàm mất mát CrossEntropy chuẩn")
print("="*70)
!python scripts/train_bilstm.py \
    --run_name bilstm_seg_w2v_ce \
    --use_word_segment true \
    --w2v_path "{W2V_FILE}" \
    --loss_type ce \
    --epochs 15 \
    --batch_size 64

print("\n" + "="*70)
print("RUN 4/6: So sánh Hàm mất mát Focal Loss đa lớp (gamma=2.0)")
print("="*70)
!python scripts/train_bilstm.py \
    --run_name bilstm_seg_w2v_focal \
    --use_word_segment true \
    --w2v_path "{W2V_FILE}" \
    --loss_type focal \
    --epochs 15 \
    --batch_size 64

print("\n" + "="*70)
print("RUN 5/6: Mô hình Đề xuất chính (Tách từ + PhoW2V + Weighted CrossEntropy)")
print("="*70)
!python scripts/train_bilstm.py \
    --run_name bilstm_seg_w2v_weighted_ce \
    --use_word_segment true \
    --w2v_path "{W2V_FILE}" \
    --loss_type weighted_ce \
    --epochs 15 \
    --batch_size 64

print("\n" + "="*70)
print("RUN 6/6: Đánh giá tác động khi bật Chuẩn hóa teencode / nhiễu (Noise Normalizer)")
print("="*70)
!python scripts/train_bilstm.py \
    --run_name bilstm_seg_w2v_noise_norm \
    --use_word_segment true \
    --normalize_noise true \
    --w2v_path "{W2V_FILE}" \
    --loss_type weighted_ce \
    --epochs 15 \
    --batch_size 64

In [ ]:
# 5. Tổng hợp bảng so sánh kết quả & Xuất mã LaTeX ACL
!python scripts/reproduce_results.py

# Đọc trực tiếp bảng LaTeX vừa tạo
print("\n--- MÃ BẢNG LATEX TẠO BỞI SCRIPT ---\n")
with open("results/summary_table.tex", "r", encoding="utf-8") as f:
    print(f.read())

# Đóng gói toàn bộ kết quả (.json, .tex) và checkpoint tốt nhất để tải về máy
!zip -r /kaggle/working/bilstm_artifacts.zip results/ checkpoints/bilstm/*best.pt
print("[+] Đã đóng gói thành công: /kaggle/working/bilstm_artifacts.zip")
print("[*] Bạn có thể tải file zip về máy từ thanh điều khiển bên phải (Data Explorer) của Kaggle!")